In [1]:
import sys
from pathlib import Path

# Add ../Retrieval to Python path
RETRIEVAL_PATH = Path("..") / "Retrieval"
sys.path.append(str(RETRIEVAL_PATH.resolve()))

In [2]:
import sys, os, csv
import torch, torchaudio
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dotenv import load_dotenv
from Classes.TextAdaptationModule import TextAdaptationModule
from Classes.DataRetrieval import DataRetrieval
from Classes.OpenAIClient import OpenAIClient
from Classes.GeminiClient import GeminiClient
from Classes.InstructionAnalysisModule import InstructionAnalysisModule
from IPython.display import Audio

import numpy as np
import soundfile as sf

sys.path.append(os.path.abspath("../Retrieval"))
sys.path.append(os.path.abspath("../CosyVoice"))
sys.path.append(os.path.abspath("../CosyVoice/third_party/Matcha-TTS"))

# CosyVoice imports
try:
    from modelscope import snapshot_download
    from cosyvoice.cli.cosyvoice import CosyVoice2
    from cosyvoice.utils.file_utils import load_wav
    
    model_path = snapshot_download("iic/CosyVoice2-0.5B")
    
except Exception as e:
    raise RuntimeError("Could not import CosyVoice2 / load_wav. Fix your CosyVoice install or imports.") from e

c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-28 00:57:55,684 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


failed to import ttsfrd, use wetext instead


2025-08-28 00:57:57,044 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice2-0.5B/revisions HTTP/1.1" 200 None
2025-08-28 00:57:57,448 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice2-0.5B/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-08-28 00:57:57,450 - modelscope - INFO - Creating symbolic link C:\Users\Admin\.cache\modelscope\hub\iic\iic/CosyVoice2-0___5B -> C:\Users\Admin\.cache\modelscope\hub\iic/CosyVoice2-0.5B.
2025-08-28 00:57:57,451 - modelscope - WARNING - Failed to create symbolic link C:\Users\Admin\.cache\modelscope\hub\iic\iic/CosyVoice2-0___5B -> C:\Users\Admin\.cache\modelscope\hub\iic/CosyVoice2-0.5B: [WinError 1314] A required privilege is not held by the client: 'C:\\Users\\Admin\\.cache\\modelscope\\hub\\iic\\iic\\CosyVoice2-0___5B' -> 'C:\\Users\\Admin\\.cache\\modelscope\\hub\\iic/CosyVoice2-0.5B'


In [3]:
def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def to_float32(audio: Any) -> Tuple[np.ndarray, int]:
    if isinstance(audio, tuple) and len(audio) == 2:
        arr, sr = audio
    elif isinstance(audio, dict) and "audio" in audio and "sample_rate" in audio:
        arr, sr = audio["audio"], audio["sample_rate"]
    else:
        arr, sr = audio, 16000

    arr = np.asarray(arr)
    try:
        arr = arr.detach().cpu().numpy()
    except Exception:
        pass

    if arr.dtype != np.float32:
        arr = arr.astype(np.float32)

    if arr.ndim > 1:
        if arr.shape[0] < arr.shape[-1]:
            arr = arr.T
        arr = arr.mean(axis=1)
    return arr, int(sr)

def _peak_normalize_float32(arr: np.ndarray, peak_db: float = -3.0) -> np.ndarray:
    if arr is None or arr.size == 0:
        return arr
    peak = float(np.max(np.abs(arr)))
    if peak <= 0.0:
        return arr
    target = 10 ** (peak_db / 20.0)
    return arr * (target / peak)

def save_audio_anything(chunks, out_path, default_sr=24000):
    """
    Save CosyVoice outputs into one WAV at the correct sample rate.

    Detect audio keys: audio/wav/pcm/tts_speech/speech/samples
    Detect SR keys: sample_rate/sr/tts_sr/sampleRate/sample_rate_hz/sampling_rate/rate/fs/hz
    Falls back to default_sr (pass DEFAULT_TTS_SR from the model).
    """
    if not chunks:
        return None, "No chunks returned from TTS call"

    KEY_CANDIDATES = ["audio", "wav", "pcm", "tts_speech", "speech", "samples"]
    SR_CANDIDATES  = ["sample_rate", "sr", "tts_sr", "sampleRate",
                      "sample_rate_hz", "sampling_rate", "rate", "fs", "hz"]

    audios = []
    detected_sr = None

    for ch in chunks:
        arr, sr = None, None

        if isinstance(ch, dict):
            for k in KEY_CANDIDATES:
                if k in ch and ch[k] is not None:
                    arr = ch[k]
                    break
            for k in SR_CANDIDATES:
                if k in ch and ch[k] is not None:
                    try:
                        sr = int(ch[k])
                    except Exception:
                        pass
                    break

        if arr is None:
            if isinstance(ch, tuple) and len(ch) == 2:
                arr, sr = ch  # (audio, sr)
            else:
                arr = ch

        try:
            a, s = to_float32({"audio": arr, "sample_rate": sr if sr else default_sr})
            if a is None or a.ndim == 0 or a.size == 0:
                continue
            audios.append(a)
            if sr:
                detected_sr = s
        except Exception:
            continue

    if not audios:
        return None, "No valid audio arrays found in chunks"

    final_sr = int(detected_sr if detected_sr else default_sr)
    concat = np.concatenate(audios, axis=0)
    concat = _peak_normalize_float32(concat, peak_db=-3.0)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(out_path), concat, final_sr)
    return out_path, None


In [4]:
print("[INFO] Loading CosyVoice model...")
cosyvoice = CosyVoice2(model_path, load_jit=False, load_trt=False, load_vllm=False, fp16=False)
print("[INFO] CosyVoice ready.")

DEFAULT_TTS_SR = (
    getattr(cosyvoice, "tts_sample_rate", None)
    or getattr(cosyvoice, "sample_rate", None)
    or 24000   # fallback
)
print("DEFAULT_TTS_SR =", DEFAULT_TTS_SR)

# Load API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

[INFO] Loading CosyVoice model...


c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\diffusers\models\lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-08-28 00:58:00,754 INFO input frame rate=25
c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\Admin\miniconda3\envs\cos

2025-08-28 00:58:04,168 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 205
2025-08-28 00:58:04,600 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-08-28 00:58:04,760 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2025-08-28 00:58:06,124 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 205
2025-08-28 00:58:06,516 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None


[INFO] CosyVoice ready.
DEFAULT_TTS_SR = 24000


In [6]:
from pathlib import Path
import json

INPUT_JSON = Path("6llamaA_2_prompts_with_results_llama3.1_8b_standard.json")
OUT_DIR = Path("6_Adapted_Llama_A")
rows = []
skipped = 0

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"[INFO] Loaded {len(data)} scenarios from {INPUT_JSON.name}")

for scen_idx, scenario in enumerate(data):
    for adapt_idx, adaptation in enumerate(scenario.get("adaptations", [])):
        uid = f"S{scen_idx:02d}_A{adapt_idx:02d}"

        results = adaptation.get("results", {})
        adapted_text = results.get("adapted_text", {}).get("llama3.1:8b", "").strip()
        audio_info = results.get("audio_path", {}).get("llama3.1:8b", [])
        speaker_info = results.get("inferred_speaker_info", {})

        if not adapted_text or not isinstance(audio_info, list) or len(audio_info) != 2:
            skipped += 1
            print(f"[WARN] Skipping {uid}: missing adapted_text or audio_info.")
            continue

        rows.append({
            "uid": uid,
            "adapted_text": adapted_text,
            "ref_audio": f"../{audio_info[0]}",
            "ref_transcript": audio_info[1],
            "speaker_info": speaker_info,
        })

print(f"[INFO] Loaded {len(rows)} rows from {INPUT_JSON.name}")
print(f"[INFO] Skipped {skipped} invalid adaptation rows.")


[INFO] Loaded 30 scenarios from 6llamaA_2_prompts_with_results_llama3.1_8b_standard.json
[INFO] Loaded 3600 rows from 6llamaA_2_prompts_with_results_llama3.1_8b_standard.json
[INFO] Skipped 0 invalid adaptation rows.


In [ ]:
import numpy as np
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
import os, csv, traceback

INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000
MANIFEST_PATH = OUT_DIR / "6_Adapted_Llama_A.tsv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "uid",
    "adapted_text",
    "ref_audio",
    "ref_transcript",
    "out_path",
    "error"
]

with open(MANIFEST_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, delimiter="\t", fieldnames=fieldnames)
    writer.writeheader()

    for i, row in enumerate(rows, start=1):
        uid = row.get("uid") or f"row{i:04d}"
        adapted_text = row.get("adapted_text", "").strip()
        ref_audio = row.get("ref_audio", "").strip()
        ref_transcript = row.get("ref_transcript", "").strip()

        print(f"\n=== {i:04d}/{len(rows)} {uid} ===")
        print("Text        :", adapted_text)
        print("Ref audio   :", ref_audio)
        
        # Get metadata from inferred_speaker_info
        speaker_info = row.get("speaker_info", {})

        scen_idx = int(uid.split("_")[0][1:])   # "S00" → 0
        adapt_idx = int(uid.split("_")[1][1:])  # "A00" → 0

        accent = speaker_info.get("accent", "UNK").upper()
        gender = speaker_info.get("gender", "U").upper()

        age = speaker_info.get("age", "UNK")
        if isinstance(age, str):
            age = age.replace(" ", "").replace("[", "").replace("]", "").split(",")[0]
        elif isinstance(age, list) and len(age) > 0:
            age = str(age[0])
        elif isinstance(age, (int, float)):
            age = str(int(age))
        else:
            age = "UNK"

        err_msg = ""
        saved_path = ""

        try:
            # Load reference audio
            prompt_tensor = load_wav(ref_audio, INPUT_SAMPLE_RATE)

            # Register the reference as a zero-shot speaker
            cosyvoice.add_zero_shot_spk(ref_transcript, prompt_tensor, uid)

            # Generate speech using the registered speaker ID
            chunks = list(
                cosyvoice.inference_zero_shot(
                    adapted_text,    # text to speak
                    "",              # prompt_text (ignored in your case)
                    "",              # empty string for legacy speaker label
                    zero_shot_spk_id=uid,
                    stream=False
                )
            )

            if not chunks:
                raise RuntimeError("No chunks returned from inference.")

            # Save output
            out_fname = f"{i:04d}_sc{scen_idx:03d}_a{accent}_g{gender}_age{age}_{adapt_idx}.wav"
            out_path = OUT_DIR / out_fname
            saved_path, err = save_audio_anything(chunks, out_path, default_sr=DEFAULT_TTS_SR)
            if err:
                raise RuntimeError(err)

            print("Saved ->", saved_path)

        except Exception as e:
            err_msg = f"{type(e).__name__}: {e}"
            print("[ERR]", err_msg)
            traceback.print_exc()

        writer.writerow({
            "uid": uid,
            "adapted_text": adapted_text,
            "ref_audio": ref_audio,
            "ref_transcript": ref_transcript,
            "out_path": str(saved_path) if saved_path else "",
            "error": err_msg,
        })

print(f"\n[MANIFEST] Wrote: {MANIFEST_PATH}")


In [33]:
# # Sanity snapshot: planned vs saved vs failed

# from pathlib import Path
# import csv, json

# INPUT_JSON = Path("./2_adapted_text_generation/1_prompts_with_speaker_info_noimplicit_final.json")
# MANIFEST_PATH = Path("./mass_generation_baseline_zeroshot_framework_prompt/zeroshot_manifest.tsv")

# # How many scenarios, target adaptations, and how many are valid (non-[ERROR])?
# with open(INPUT_JSON, "r", encoding="utf-8") as f:
#     data = json.load(f)

# num_scenarios = len(data)
# target_total = sum(len(s.get("adaptations") or []) for s in data)

# # Count invalids
# invalids = []
# valid_total = 0
# for scen_idx, scen in enumerate(data, start=1):
#     txt = (scen.get("standard_sentence") or "").strip()
#     if len(txt) >= 2 and txt[0] == txt[-1] and txt[0] in ("'", '"'):
#         txt = txt[1:-1]
#     for adapt_idx, adapt in enumerate(scen.get("adaptations") or [], start=1):
#         instr = (adapt.get("explicit_instruction") or "").strip()
#         if not txt or not instr or instr.startswith("[ERROR]"):
#             invalids.append((scen_idx, adapt_idx, instr))
#         else:
#             valid_total += 1

# print(f"[SCENARIOS] {num_scenarios}")
# print(f"[ADAPTATIONS in JSON] {target_total}")
# print(f"[INVALID skipped rows] {len(invalids)} (expected 3)")
# print(f"[VALID rows your loop runs] {valid_total}")

# # What actually got saved
# saved = 0
# failed = 0
# missing_on_disk = 0

# if MANIFEST_PATH.exists():
#     with open(MANIFEST_PATH, "r", encoding="utf-8", newline="") as f:
#         r = csv.DictReader(f, delimiter="\t")
#         rows = list(r)

#     for row in rows:
#         err = (row.get("error") or "").strip()
#         outp = (row.get("output_path") or "").strip()
#         if err:
#             failed += 1
#         elif outp:
#             if Path(outp).exists():
#                 saved += 1
#             else:
#                 missing_on_disk += 1

#     print(f"[MANIFEST] rows: {len(rows)}")
#     print(f"[SAVED files] {saved}")
#     print(f"[FAILED (error set)] {failed}")
#     print(f"[MISSING on disk despite path in manifest] {missing_on_disk}")
# else:
#     print(f"[WARN] Manifest not found at {MANIFEST_PATH}")
